# Conversational UI Chatbot App with Gemini, LangChain and Chainlit

I am building an advanced Gemini Conversational UI-based chatbot using LangChain and Chainlit with the following features:

- Custom Landing Page
- Conversational memory
- Result streaming capabilities (Real-time output)

## Install App and LLM dependencies

In [1]:
!pip install langchain==0.1.12
!pip install langchain-google-genai==0.0.11
!pip install chainlit==1.0.401
!pip install pyngrok==7.1.5

  Using cached protobuf-4.25.9-cp37-abi3-manylinux2014_x86_64.whl.metadata (541 bytes)
Using cached protobuf-4.25.9-cp37-abi3-manylinux2014_x86_64.whl (295 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.6
    Uninstalling protobuf-6.33.6:
      Successfully uninstalled protobuf-6.33.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.41.1 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.41.1 which is incompatible.
grain 0.2.17 requires protobuf>=5.28.3, but you hav

  Using cached protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
Using cached protobuf-6.33.6-cp39-abi3-manylinux2014_x86_64.whl (323 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.9
    Uninstalling protobuf-4.25.9:
      Successfully uninstalled protobuf-4.25.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.4.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 6.33.6 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.41.1 which is incompatible.
google-adk 1.29.0 requires anyio<5.0.0,>=4.9.0, but you have anyio 3.7.1 which is incompatible.
google-adk 1.29.0 requires fastapi<1.0.0,>=0.124.1, but you h

## Load Gemini API Credentials

In [2]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [3]:
import yaml

with open('gemini_key.yml', 'r') as file:
    api_creds = yaml.safe_load(file)

In [4]:
api_creds.keys()

dict_keys(['gemini_key'])

In [5]:
import os

os.environ['GEMINI_API_KEY'] = api_creds['gemini_key']

## This is the app code, and it is stored in a PY file.

In [6]:
# that will write all the code below it to the python file app.py
# deploy this app.py file on the cloud server where colab is running
# if you have own server just write the code in app.py and deploy it directly
%%writefile app.py

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.memory import ConversationBufferWindowMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain.schema.runnable.config import RunnableConfig
from langchain.schema import StrOutputParser
from operator import itemgetter
import chainlit as cl

@cl.on_chat_start
# this function is called when the app starts for the first time
async def when_chat_starts():

  # Load a connection to Gemini LLM
  gemini = ChatGoogleGenerativeAI(model="gemini-pro",
                                      convert_system_message_to_human=True, temperature=0.1,streaming=True)

  # Add a basic system prompt for LLM behavior
  SYS_PROMPT = """
               Act as a helpful assistant and answer questions to the best of your ability.
               Do not make up answers.
               """

  # Creating a prompt template for langchain to use history to answer user prompts
  prompt = ChatPromptTemplate.from_messages(
    [
      ("system", SYS_PROMPT),
      MessagesPlaceholder(variable_name="history"),
      ("human", "{input}"),
    ]
  )

  # Creating a memory object to store conversation history window
  memory = ConversationBufferWindowMemory(k=20,
                                          return_messages=True)

  # Creating a conversation chain
  conversation_chain = (
    RunnablePassthrough.assign(
      history=RunnableLambda(memory.load_memory_variables)
      |
      itemgetter("history")
    )
    |
    prompt
    |
    gemini
    |
    StrOutputParser() # to parse the output to show on UI
  )
  # Set session variables to be accessed when user enters prompts in the app
  cl.user_session.set("chain", conversation_chain)
  cl.user_session.set("memory", memory)


@cl.on_message
# this function is called whenever the user sends a prompt message in the app
async def on_user_message(message: cl.Message):

  # get the chain and memory objects from the session variables
  chain = cl.user_session.get("chain")
  memory = cl.user_session.get("memory")

  # this will store the response from Gemini LLM
  gemini_message = cl.Message(content="")

  # Stream the response from Gemini and show in real-time
  async for chunk in chain.astream(
    {"input": message.content},
    config=RunnableConfig(callbacks=[cl.LangchainCallbackHandler()]),
  ):
      await gemini_message.stream_token(chunk)
  # Finish displaying the full response from Gemini
  await gemini_message.send()
  # Store the current conversation in the memory object
  memory.save_context({"input": message.content},
                      {"output": gemini_message.content})

Overwriting app.py


## Start the app

In [7]:
!chainlit run app.py --port=8989 --watch &>./logs.txt &

## Change the Initial app screen

In [8]:
%%writefile chainlit.md

# Welcome I am AI Assistant 🤖

How can I help you today?

Writing chainlit.md


In [9]:
from pyngrok import ngrok
import yaml

# Terminate open tunnels if exist
ngrok.kill()

# Setting the authtoken
# Get authtoken from `ngrok_credentials.yml` file
with open('ngrok_credentials.yml', 'r') as file:
    NGROK_AUTH_TOKEN = yaml.safe_load(file)
ngrok.set_auth_token(NGROK_AUTH_TOKEN['ngrok_key'])

# Open an HTTPs tunnel on port XXXX which you get from your `logs.txt` file
ngrok_tunnel = ngrok.connect(8989)
print("Chainlit App:", ngrok_tunnel.public_url)

Chainlit App: https://rewire-bulldog-frostlike.ngrok-free.dev


## Remove running app processes

In [10]:
ngrok.kill()

In [11]:
!ps -ef | grep app

root           7       1  0 00:09 ?        00:00:09 /tools/node/bin/node /datalab/web/app.js
root          88       7  0 00:09 ?        00:00:09 /usr/bin/python3 /usr/local/bin/jupyter-server --debug --transport="ipc" --ip=172.28.0.12 --ServerApp.token= --port=9000 --FileContentsManager.root_dir=/ --FileContentsManager.allow_hidden=True --ServerApp.log_format="|%(levelname)s|%(message)s" --ServerApp.iopub_data_rate_limit=1e10 --MappingKernelManager.root_dir=/content
root        8362       1 75 00:42 ?        00:00:00 /usr/bin/python3 /usr/local/bin/chainlit run app.py --port=8989 --watch
root        8389    8133  0 00:42 ?        00:00:00 /bin/bash -c ps -ef | grep app
root        8391    8389  0 00:42 ?        00:00:00 grep app


In [12]:
!sudo kill -9 5617

kill: (5617): No such process
